# Generating Irish Folk Music with an LSTM

In this lab we teach a neural network to write music. The idea is simpler than it sounds: our dataset is a collection of Irish folk songs written in **ABC notation**, which is just plain text. So we can treat music generation as a **language modeling** problem at the character level: given a sequence of characters, predict the next one. Once the model is good at that, we can let it "hallucinate" brand new songs, one character at a time, and then synthesize them into audio.

Along the way we will build a **character-level LSTM** in PyTorch, add a proper **train/validation split**, **early stopping** and **checkpointing**, and track every run with **Comet ML**.

## Running this notebook on Kaggle

This notebook is designed to run on **Kaggle**, which gives us a free GPU and a clean way to store API keys. To set it up:

1. Create a new Notebook on [Kaggle](https://www.kaggle.com/code) and import this file (File > Import Notebook);
2. In the notebook settings, set **Accelerator** to a GPU (for example a T4 or P100) and make sure **Internet** is enabled;
3. Create a free account on [Comet](https://www.comet.com), which we will use to track our experiments, and copy your **API key** from your account settings;
4. Back on Kaggle, open **Add-ons > Secrets**, create a secret with label `COMET_API_KEY`, paste your key as the value, and attach it to this notebook.

The first cell reads the key through `kaggle_secrets`, so the key never appears in the code. If you run this elsewhere (Colab, local machine), just replace that block with your own way of loading the key.

One note on dependencies: the `mitdeeplearning` package is a small utility library from the open MIT deep learning course. We only use it here to download the song dataset and for a couple of plotting and audio helpers.

In [ ]:
# importing packages

!pip install comet_ml > /dev/null 2>&1
import comet_ml
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
COMET_API_KEY = user_secrets.get_secret("COMET_API_KEY")

import torch
import torch.nn as nn
import torch.optim as optim

!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
from scipy.io.wavfile import write
!apt-get install abcmidi timidity > /dev/null 2>&1

from IPython import display
import matplotlib.pyplot as plt

assert COMET_API_KEY != "", "Please insert your Comet API Key"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## The dataset

We load thousands of Irish folk songs and join them into one giant string. From that string we extract the **vocabulary**: the sorted set of every unique character that appears (letters, digits, punctuation, newlines). This is the entire "alphabet" our model will ever know. The model does not know what a note or a chord is: it only sees characters, and it will have to discover the structure of music by itself.

In [ ]:
songs = mdl.lab1.load_training_data()
songs_joined = "\n\n".join(songs)
vocab = sorted(set(songs_joined))

## From characters to numbers

Neural networks do not eat text, they eat numbers. So we build two lookup tables: `char2idx` maps every character to a unique integer id, and `idx2char` does the reverse. Together they let us move back and forth between the world of text and the world of tensors.

In [ ]:
# methods to pass from char to an id number and from an id number to char, we will use it to create a lookup table
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

With the lookup table ready, vectorizing a string is just a matter of replacing every character with its id. We apply this to the entire dataset, obtaining one long NumPy array of integers.

In [ ]:
def vectorize_string(string):
    vector = np.array([char2idx[char] for char in string])
    return vector

vectorized_songs = vectorize_string(songs_joined)
assert isinstance(vectorized_songs, np.ndarray)

## Building training batches

Here is the trick that makes the whole lab work. For every training example we take a random slice of the dataset of length `seq_length` as **input**, and the same slice **shifted one position to the right** as **target**. Why? Because the task we want to learn is "predict the next character": at every position, the correct answer is exactly the character that follows.

For example, if the text is `Hello`, the input is `Hell` and the target is `ello`: given `H` predict `e`, given `He` predict `l`, and so on. We stack `batch_size` of these random slices together so the GPU can process them in parallel.

In [ ]:
def get_batch(vectorized_songs, seq_length, batch_size):
    # let's calculate the vectorized songs string max index to prevent OOB EX
    n = vectorized_songs.shape[0] - 1
    # let's choose a random idx for choose the batch start index
    idx = np.random.choice(n - seq_length, batch_size)
    # we take a number of input slices equal to batch_size
    input_batch = [vectorized_songs[i: i + seq_length] for i in idx]
    # same logic for the target, but shifted one position to the right, because it is the prediction
    output_batch = [vectorized_songs[i+ 1: i + seq_length + 1] for i in idx]

    x_batch = torch.tensor(input_batch, dtype=torch.long).to(device)
    y_batch = torch.tensor(output_batch, dtype=torch.long).to(device)

    return x_batch, y_batch

test_args = (vectorized_songs, 2, 10)
x_batch, y_batch = get_batch(*test_args)
print(vectorized_songs.shape)
print(x_batch.shape)
print(y_batch.shape)
example_idx = 0
print("X (Input) :", x_batch[example_idx].tolist())
print("Y (Target):", y_batch[example_idx].tolist())

## The model: Embedding, LSTM, Linear

Our network is a stack of three pieces:

- **Embedding**: turns each character id into a dense vector of `embedding_dim` numbers. Instead of a meaningless integer, each character gets a learned representation, and similar characters can end up with similar vectors;
- **LSTM**: the heart of the model. It reads the sequence one step at a time while carrying an internal **state**, a sort of memory that lets it remember information from many characters ago (essential for music: the ending of a phrase depends on how it started). We stack `num_layers` LSTM layers with dropout between them to reduce overfitting;
- **Linear**: the final head. It projects the LSTM output to a vector with one score (**logit**) per character in the vocabulary: our prediction of what comes next.

The `init_hidden` method builds a zeroed initial state, and `forward` optionally returns the final state, which we will need later during generation to keep the "memory" alive between one character and the next.

In [ ]:
# let's define our recurrent neural network (RNN) model

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super(LSTMModel, self).__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        # to convert the text in a vector of numbers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # the LSTM analyzes the input vectors and maintains an internal state, allowing the network to remember long term information
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, dropout=0.2, batch_first=True)
        # takes the hidden state as input and transforms it into the vector with the score for each char
        self.fc = nn.Linear(hidden_size, vocab_size)

    def init_hidden(self, batch_size, device):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
            torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device))

    def forward(self, x, state=None, return_state=False):
        x = self.embedding(x)

        if state is None:
          state = self.init_hidden(x.size(0), x.device)
        out, state = self.lstm(x, state)

        out = self.fc(out)
        return out if not return_state else (out, state)

## The loss function

This is a classification problem: at every position, "which of the vocabulary characters comes next?". So we use **cross entropy**. The only subtlety is the shape: the model outputs a 3D tensor (batch, sequence, vocabulary), while `CrossEntropyLoss` wants a 2D one. We simply flatten batch and sequence together, so every position in every sequence counts as one independent prediction.

In [ ]:
cross_entropy = nn.CrossEntropyLoss()
def compute_loss(labels, logits):
    batched_labels = labels.view(-1)
    batched_logits = logits.view(-1, logits.size(-1))
    loss = cross_entropy(batched_logits, batched_labels)
    return loss

## Train/validation split

We keep the last 10% of the data aside as a **validation set**. The model never trains on it, so measuring the loss there tells us how well it generalizes to music it has never seen. If the training loss keeps dropping while the validation loss starts rising, the model is **overfitting**: memorizing the training songs instead of learning the style.

In [ ]:
split = int(0.9 * len(vectorized_songs))
train_data = vectorized_songs[:split]
val_data = vectorized_songs[split:]

The validation loss function averages the loss over a few random validation batches. Note the two safety switches: `model.eval()` disables dropout, and `torch.no_grad()` disables gradient tracking, since here we only want to measure, not learn. We switch back to `model.train()` at the end.

In [ ]:
def compute_val_loss(num_batches=20):
    model.eval()
    with torch.no_grad():
        losses = []
        for _ in range(num_batches):
            x, y = get_batch(val_data, params["seq_length"], params["batch_size"])
            y_hat = model(x)
            losses.append(compute_loss(y, y_hat).item())
    model.train()
    return np.mean(losses)

A small helper to plot the training and validation curves live during training. The validation loss is computed every 10 iterations, so we scale its x axis accordingly.

In [ ]:
def plot_losses(history, val_history):
    display.clear_output(wait=True)
    plt.figure(figsize=(10, 4))
    plt.plot(history, label="train loss")
    val_iters = [i * 10 for i in range(len(val_history))]
    plt.plot(val_iters, val_history, label="val loss")
    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

## Hyperparameters

All the knobs in one place:

- `seq_length = 300`: how many characters of context the model sees per example;
- `batch_size = 256`: how many sequences we process in parallel;
- `embedding_dim = 256` and `hidden_size = 1024`: the capacity of the model;
- `num_layers = 2`: two stacked LSTM layers;
- `learning_rate = 5e-3`: the step size of the optimizer.

Feel free to experiment: these are exactly the parameters worth tweaking to see how the loss curves react.

In [ ]:
vocab_size = len(vocab)

params = dict(
    num_training_iteractions = 500,
    batch_size = 256,
    seq_length = 300,
    learning_rate = 5e-3,
    embedding_dim = 256,
    hidden_size = 1024,
    num_layers = 2
)

torch.cuda.empty_cache()

## Experiment tracking with Comet

Every training run gets logged to **Comet ML**: hyperparameters, loss curves, and at the end even the generated audio files. This is a habit worth building early: when you start tweaking hyperparameters, being able to compare runs side by side is worth gold.

In [ ]:
def create_experiment():
  if 'experiment' in locals():
    experiment.end()

  experiment = comet_ml.Experiment(api_key=COMET_API_KEY, project_name="irish-folk-music-generation")

  for param, value in params.items():
    experiment.log_parameter(param, value)

  return experiment

## The training step

We instantiate the model, move it to the GPU, and use **Adam** as optimizer. A single training step is the classic PyTorch loop: reset gradients with `zero_grad()`, run the **forward pass**, compute the loss, call `loss.backward()` to backpropagate, and let `optimizer.step()` update the weights.

In [ ]:
model = LSTMModel(vocab_size, params["embedding_dim"], params["hidden_size"], num_layers=params["num_layers"])
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

def train_step(x, y):
    model.train()
    optimizer.zero_grad()
    x = x.to(device)
    y = y.to(device)
    y_hat = model(x)
    loss = compute_loss(y, y_hat)
    loss.backward()
    optimizer.step()
    return loss

We prepare a directory for **checkpoints**: every time the validation loss improves we will save the model weights there, so at the end we can restore the best version, not just the last one.

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")
os.makedirs(checkpoint_dir, exist_ok=True)

## The training loop, with early stopping

The loop puts everything together: get a batch, do a training step, log the loss. Every 10 iterations we also measure the **validation loss**, and here is where the interesting logic lives:

- if the validation loss improved, we save a checkpoint and reset the patience counter;
- if it did not improve for `patience = 5` consecutive checks, we stop training early (**early stopping**). Going on would only mean overfitting.

When the loop ends, we reload the best checkpoint. This way the model we keep is the one that generalized best, not the one from the last iteration.

In [ ]:
history = []
val_history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel="Iterations", ylabel="Loss")
experiment = create_experiment()

best_val_loss = float('inf')
patience = 5
no_improve = 0

if hasattr(tqdm, '_instances'): tqdm._instances.clear()
    
for iter in tqdm(range(params["num_training_iteractions"]), unit="it", leave=True):
    x_batch, y_batch = get_batch(train_data, params["seq_length"], params["batch_size"])

    loss = train_step(x_batch, y_batch)

    experiment.log_metric("loss", loss.item(), step=iter)

    history.append(loss.item())
    plot_losses(history, val_history)

    if iter % 10 == 0:
        val_loss = compute_val_loss()
        val_history.append(val_loss)
        experiment.log_metric("val_loss", val_loss, step=iter)
    
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_prefix)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at iter {iter}, best val_loss: {best_val_loss:.4f}")
                break

experiment.flush()
model.load_state_dict(torch.load(checkpoint_prefix))

## Generating new music

Now the fun part. Generation is **autoregressive**: we feed the model a starting string, it predicts a probability distribution over the next character, we pick one, append it to the input, and repeat.

One key detail: we do not always pick the most likely character (that would be greedy and repetitive). Instead we **sample** from the distribution with `torch.multinomial`, so the model can surprise us. Note also that we carry the LSTM `state` forward at every step: that is the model's memory of everything generated so far.

In [ ]:
def generate_text(model, start_string, generation_length=1000):

  input_idx = [char2idx[s] for s in start_string]
  input_idx = torch.tensor([input_idx], dtype=torch.long).to(device)

  state = model.init_hidden(input_idx.size(0), device)

  text_generated = []
  tqdm._instances.clear()
  with torch.no_grad():
    for i in tqdm(range(generation_length)):
      predictions, state = model(input_idx, state, return_state=True)
      predictions = predictions.squeeze(0)

      input_idx = torch.multinomial(torch.softmax(predictions, dim=-1), num_samples=1)

      text_generated.append(idx2char[input_idx].item())

  return (start_string + ''.join(text_generated))

## From text to audio

The generated text should contain valid songs in ABC notation. We extract every well-formed song snippet, synthesize it into a waveform, play it inline, save it as a `.wav` file, and log it to Comet so we can listen to our model's compositions directly from the experiment page.

Not every snippet will be syntactically valid, and that is fine: the model learned the format purely from examples. Listen to a few outputs, then try playing with the hyperparameters or the generation length and see how the music changes.

In [ ]:
generated_text = generate_text(model, start_string="X", generation_length=1000)
generated_songs = mdl.lab1.extract_song_snippet(generated_text)

for i, song in enumerate(generated_songs):
  # Synthesize the waveform from a song
  waveform = mdl.lab1.play_song(song)

  # if it's a valid song (correct syntax), let's play it!
  if waveform:
    ipythondisplay.display(waveform)

    numeric_data = np.frombuffer(waveform.data, dtype=np.int16)
    wav_file_path = f"output_{i}.wav"
    write(wav_file_path, 88200, numeric_data)

    # save your song to the Comet interface, you can access it there
    experiment.log_asset(wav_file_path)